# Modeling Workflow
### Woojin Park, Korea University
### Copyright 2026, Korea University

## Requirements

In [ ]:
# Optional dependency install for a fresh environment.
# !pip install -r ../requirements.txt

## Imports

In [ ]:
train_df, validation_df, test_df = prepare_modeling_splits(
    input_path=INPUT_PATH,
    output_dir=PROJECT_ROOT / 'data' / '03_modeling_inputs',
    run_name=RUN_NAME,
    text_column=TEXT_COLUMN,
    samples_per_class=SAMPLES_PER_CLASS,
    random_state=RANDOM_STATE,
    export_csv=True,
)

print('train:', train_df.shape)
print('validation:', validation_df.shape)
print('test:', test_df.shape)
print('')
print('Train label counts:')
print(train_df['label_str'].value_counts().sort_index())
print('')
print('Saved modeling inputs to:', MODELING_INPUT_DIR)

## Configuration

In [ ]:
# Change this value for larger experiments, for example 20000 or 40000.
SAMPLES_PER_CLASS = 1000
RANDOM_STATE = 42
TEXT_COLUMN = 'title_with_selftext_cleaned'

INPUT_PATH = PROJECT_ROOT / 'data' / '02_preprocessing_outputs' / 'final_preprocessed_df.csv'
RUN_NAME = f'sample_{SAMPLES_PER_CLASS}_per_class'
MODELING_INPUT_DIR = PROJECT_ROOT / 'data' / '03_modeling_inputs' / RUN_NAME

CLASS_NAMES = [ID_TO_LABEL[i] for i in sorted(ID_TO_LABEL)]
CLASS_NAMES
MODEL_CHECKPOINT = 'distilbert-base-uncased'
OUTPUT_DIR = PROJECT_ROOT / 'models' / f'distilbert_{RUN_NAME}'

## Load, Sample, and Split Dataset

In [ ]:
train_df, validation_df, test_df = prepare_modeling_splits(
    input_path=INPUT_PATH,
    output_dir=PROJECT_ROOT / 'data' / '03_modeling_inputs',
    run_name=RUN_NAME,
    text_column=TEXT_COLUMN,
    samples_per_class=SAMPLES_PER_CLASS,
    random_state=RANDOM_STATE,
    export_csv=True,
)

print('train:', train_df.shape)
print('validation:', validation_df.shape)
print('test:', test_df.shape)
print('')
print('Train label counts:')
print(train_df['label_str'].value_counts().sort_index())
print('')
print('Saved modeling inputs to:', MODELING_INPUT_DIR)

## Convert to HuggingFace Dataset

In [ ]:
features = Features({'text': Value('string'), 'label': ClassLabel(names=CLASS_NAMES)})

def to_hf_dataset(df):
    return Dataset.from_pandas(df[['text', 'label']], preserve_index=False).cast(features)

dataset = DatasetDict({
    'train': to_hf_dataset(train_df),
    'validation': to_hf_dataset(validation_df),
    'test': to_hf_dataset(test_df),
})

dataset

## Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, average='weighted', zero_division=0),
        'recall': recall_score(labels, preds, average='weighted', zero_division=0),
        'f1': f1_score(labels, preds, average='weighted'),
    }


def plot_normalized_confusion_matrix(trainer, encoded_dataset, split='test'):
    predictions = trainer.predict(encoded_dataset[split])
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = predictions.label_ids
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(cmap='Blues', values_format='.2f', ax=ax, colorbar=False)
    plt.title(f'Normalized confusion matrix ({split})')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=512)

encoded_dataset = dataset.map(tokenize, batched=True)
encoded_dataset = encoded_dataset.remove_columns(['text'])
encoded_dataset.set_format('torch')
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Fine-Tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(CLASS_NAMES),
    id2label={i: label for i, label in enumerate(CLASS_NAMES)},
    label2id={label: i for i, label in enumerate(CLASS_NAMES)},
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Evaluation

In [ ]:
validation_metrics = trainer.evaluate(encoded_dataset['validation'])
test_metrics = trainer.evaluate(encoded_dataset['test'])
print('validation:', validation_metrics)
print('test:', test_metrics)
plot_normalized_confusion_matrix(trainer, encoded_dataset, split='test')

## Save Model

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print('Saved model to:', OUTPUT_DIR)